In [ ]:
# Import the libraries required for environment variables, JSON handling, and the OpenAI client.
import os
import json

from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
# Load the OpenAI API key and initialize the OpenAI client.
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found, kindly recheck this.")
else:
    print("API key found.")

openai = OpenAI()

In [ ]:
# Define a realistic job description that the analyzer will process.
job_description = """
AI Automation Specialist

Company: GrowthFlow

We are looking for an AI Automation Specialist to design, build, and maintain
AI-powered workflows that improve business operations.

Responsibilities:
- Build and maintain automation workflows using n8n.
- Integrate AI models and external APIs into business processes.
- Automate repetitive marketing and operational tasks.
- Monitor workflows and troubleshoot automation failures.
- Work with teams to identify processes that can be improved through automation.

Required Skills:
- Experience with n8n or similar workflow automation platforms.
- Knowledge of APIs and webhooks.
- Experience working with AI and LLM applications.
- Strong understanding of Python.
- Experience with JSON and data processing.

Preferred Skills:
- Experience with OpenAI APIs.
- Experience with databases.
- Familiarity with Docker.
- Understanding of RAG systems.

Experience:
2+ years of experience in automation, software development, AI engineering,
or a related field.

Education:
Bachelor's degree in Computer Science, Engineering, or a related technical field.

Location:
Remote.

Employment Type:
Full-time.
"""

In [ ]:
# Define the instructions for extracting structured information from the job description.
job_prompt = """
You are a job description intelligence assistant.

Analyze the job description and return valid JSON using exactly this structure:

{
    "job_title": "",
    "company": "",
    "employment_type": "",
    "location": "",
    "responsibilities": [],
    "required_skills": [],
    "preferred_skills": [],
    "experience_requirement": "",
    "education_requirement": "",
    "technologies": []
}

Only extract information explicitly present in the job description.

Do not invent information.

If information is not available, use an empty string or empty list.

Return only valid JSON.
"""

In [ ]:
# Define the required fields and validate the structured information returned by the LLM.
job_fields = [
    "job_title",
    "company",
    "employment_type",
    "location",
    "responsibilities",
    "required_skills",
    "preferred_skills",
    "experience_requirement",
    "education_requirement",
    "technologies"
]


def validate_job(data):
    if not data:
        return False

    return all(field in data for field in job_fields)

In [ ]:
# Send the job description to the LLM, parse the JSON response, and validate the extracted information.
def analyze_job(job_description):
    messages = [
        {"role": "system", "content": job_prompt},
        {"role": "user", "content": job_description}
    ]

    try:
        response = openai.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages
        )

        result = response.choices[0].message.content
        data = json.loads(result)

        if not validate_job(data):
            print("The LLM response is missing required fields.")
            return None

        return data

    except json.JSONDecodeError:
        print("The LLM returned invalid JSON.")
        return None

    except Exception as error:
        print(f"An error occurred: {error}")
        return None

In [ ]:
# Run the job description analyzer and display the extracted information.
result = analyze_job(job_description)

for field, value in result.items():
    print(f"\n{field.upper()}:")
    print(value)

In [ ]:
# Analyze the extracted job information and identify the main requirements a candidate should focus on.
def analyze_requirements(data):
    return {
        "core_skills": data["required_skills"],
        "additional_skills": data["preferred_skills"],
        "experience": data["experience_requirement"],
        "education": data["education_requirement"],
        "technologies": data["technologies"]
    }


requirements = analyze_requirements(result)

for field, value in requirements.items():
    print(f"\n{field.upper()}:")
    print(value)

In [ ]:
# Create test cases containing different job descriptions and expected extracted job titles.
evaluation_jobs = [
    {
        "expected_title": "AI Automation Engineer",
        "job": """
        We are hiring an AI Automation Engineer to build AI workflows,
        integrate APIs, and automate business processes using Python and n8n.
        """
    },
    {
        "expected_title": "Machine Learning Engineer",
        "job": """
        We are looking for a Machine Learning Engineer to develop and deploy
        machine learning models using Python, PyTorch, and scikit-learn.
        """
    },
    {
        "expected_title": "Backend Developer",
        "job": """
        Our company is hiring a Backend Developer to build APIs and backend
        services using Python, FastAPI, and PostgreSQL.
        """
    },
    {
        "expected_title": "Data Analyst",
        "job": """
        We need a Data Analyst to analyze business data, create reports,
        and build dashboards using SQL, Excel, and Power BI.
        """
    }
]

In [ ]:
# Evaluate the analyzer while safely handling invalid or failed LLM responses.
correct = 0
failed = 0

for item in evaluation_jobs:
    result = analyze_job(item["job"])

    if result is None:
        print(f"Expected: {item['expected_title']}")
        print("Predicted: FAILED")
        print("The analyzer returned no valid result.")
        print("-" * 40)
        failed += 1
        continue

    predicted = result["job_title"]

    print(f"Expected: {item['expected_title']}")
    print(f"Predicted: {predicted}")
    print("-" * 40)

    if predicted.lower() == item["expected_title"].lower():
        correct += 1

accuracy = correct / len(evaluation_jobs)

print(f"Successful predictions: {correct}")
print(f"Failed predictions: {failed}")
print(f"Job Title Extraction Accuracy: {accuracy:.2%}")